[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Link and BackLink &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's boot cell and the second defines `Maker` and `Product` and fills
them. Run both first, then any task in any order.


In [1]:
import os
import random
import subprocess
import sys
import time
import warnings
from typing import List, Optional
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import beanie
import pymongo
from beanie import BackLink, Document, Link, WriteRules, init_beanie
from pydantic import Field
from pymongo import AsyncMongoClient

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


In [2]:
class Maker(Document):
    name: str
    country: str = "unknown"

    class Settings:
        name = "makers"


class Product(Document):
    sku: str
    price: float
    maker: Link[Maker]

    class Settings:
        name = "linked"


client = AsyncMongoClient(URI)
await init_beanie(database=client.get_default_database(), document_models=[Maker, Product])
await Maker.delete_all()
await Product.delete_all()

aster = Maker(name="Aster", country="IE")
belden = Maker(name="Belden", country="PT")
await aster.insert()                                                # one at a time, because
await belden.insert()                                               # insert_many leaves id unset
await Product.insert_many([Product(sku="P-1", price=10.0, maker=aster),
                           Product(sku="P-2", price=20.0, maker=belden)])
print("ready:", await Product.find_all().count(), "products")


ready: 2 products


**1.** Two models and one reference.


In [3]:
print("makers:  ", [(m.name, m.country) for m in await Maker.find_all().to_list()])
print("products:", [(p.sku, type(p.maker).__name__) for p in await Product.find_all().to_list()])


makers:   [('Aster', 'IE'), ('Belden', 'PT')]
products: [('P-1', 'Link'), ('P-2', 'Link')]


The products are there and their `maker` fields are `Link` objects, because a plain `find` does not
resolve them.


**2.** What the field really is.


In [4]:
product = await Product.find_one(Product.sku == "P-1")
print("type: ", type(product.maker).__name__)
print("truthy:", bool(product.maker), "| is None:", product.maker is None)
print("so neither a None check nor an if will catch it")


type:  Link
truthy: True | is None: False
so neither a None check nor an if will catch it


That is what makes the `AttributeError` surprising: the field is populated with something, and the
something is not the document.


**3.** Resolving one.


In [5]:
product = await Product.find_one(Product.sku == "P-1")
await product.fetch_link(Product.maker)
print(type(product.maker).__name__, "|", product.maker.name, product.maker.country)


Maker | Aster IE


One extra round trip for one document. Fine here, and the shape to avoid inside a loop.


**4.** Resolving a whole query.


In [6]:
rows = await Product.find_all(fetch_links=True).sort(Product.sku).to_list()
for product in rows:
    print(f"  {product.sku}  {product.maker.name} ({product.maker.country})")
print("one query, with the join done by the server")


  P-1  Aster (IE)
  P-2  Belden (PT)
one query, with the join done by the server


`$lookup` stages were added to the query, so the makers came back with the products rather than in
two more trips.


**5.** Filtering through the link.


In [7]:
found = await Product.find(Product.maker.country == "PT", fetch_links=True).to_list()
print("with fetch_links:   ", [p.sku for p in found])

missed = await Product.find(Product.maker.country == "PT").to_list()
print("without fetch_links:", [p.sku for p in missed], "<- legal, and can never match")


with fetch_links:    ['P-2']
without fetch_links: [] <- legal, and can never match


Without the join there is no `country` under `maker`, only a DBRef, so the filter has nothing to
match and says so by returning nothing.


**6.** A link to something unsaved.


In [8]:
unsaved = Maker(name="Corvid", country="ES")

try:
    await Product(sku="P-3", price=3.0, maker=unsaved).insert()
except beanie.exceptions.DocumentWasNotSaved as error:
    print("as it is:", type(error).__name__ + ":", error)

await Product(sku="P-3", price=3.0,
              maker=Maker(name="Corvid", country="ES")).insert(link_rule=WriteRules.WRITE)
print("with WriteRules.WRITE, the maker exists:",
      await Maker.find_one(Maker.name == "Corvid") is not None)

await Maker.delete_all()
await Product.delete_all()
await client.close()


as it is: DocumentWasNotSaved: Can not create dbref without id
with WriteRules.WRITE, the maker exists: True


A `Link` needs an id and the unsaved maker has none. Beanie refuses rather than writing a reference
to nothing, which is more than a hand-rolled id field would have done for you.


---

&#8592; **Back to:** [Link and BackLink](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/14-link-and-backlink.ipynb)  &nbsp;&middot;&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
